# CAVAanalytics: Pakistan climate-risk introduction

This short practical shows how **CAVAanalytics** turns remote CORDEX-CORE climate data into a multi-model climate indicator and an interpretable map. The worked example asks: **How could the number of April–June days above 35 °C change in Pakistan?**

The live path takes about 10 minutes when the data server and network are responsive.

## 1. Create one Mamba environment for R and Python

Run these commands in a terminal, not in an R cell. The tested local environment resolved to **R 4.5.3**, **Python 3.12.13**, **OpenJDK 25.0.2**, **CAVAanalytics 4.0.6**, and **cavapy 2.0.7**. R 4.6.1 can be used when it is available from your configured channel.

```bash
mamba create -n cavapk -c conda-forge \
  python=3.12 r-base=4.5 \
  r-rjava r-pak r-terra r-units r-remotes \
  jupyterlab ipykernel r-irkernel

mamba activate cavapk
R -q -e 'remotes::install_github("un-fao/CAVAanalytics")'
python -m pip install cavapy
```

## 2. Define the Pakistan analysis

Pakistan belongs to the CORDEX **WAS-22** domain. We use the ready-to-use `CORDEX-CORE-BC` archive, which has already been bias-corrected against ERA5 with the ISIMIP methodology. Therefore, later analysis calls must keep `bias.correction = FALSE`.


In [ ]:
library(CAVAanalytics)

COUNTRY <- "Pakistan"
DOMAIN <- "WAS-22"
DATASET <- "CORDEX-CORE-BC"
VARIABLE <- "tasmax"
BASELINE_YEARS <- 1991:2000
FUTURE_YEARS <- 2030:2050
PRE_MONSOON_MONTHS <- 4:6
HEAT_THRESHOLD_C <- 35


SyntaxError: illegal target for annotation (3405100604.py, line 5)

## 4. Load historical and projected maximum temperature

This is the slowest cell. CAVAanalytics discovers the available simulations, converts Pakistan to a bounding box, streams the required grid cells, harmonizes the model data, and assembles the ensemble.


In [ ]:
pakistan_tasmax <- load_data(
      path.to.data = DATASET,
      country = COUNTRY,
      variable = VARIABLE,
      years.hist = BASELINE_YEARS,
      years.proj = FUTURE_YEARS,
      domain = DOMAIN,
      n.sessions = 6
    )

saveRDS(pakistan_tasmax, "PAK/data/pakistan_tasmax.rds")


In [ ]:
pakistan_tasmax=readRDS("PAK/data/pakistan_tasmax.rds")


## 5. Calculate a climate-change indicator

The indicator counts April–June days above 35 °C for each model and compares `2030–2050` with `1991–2000`. A grid cell is marked as agreeing when at least 80% of available models agree on the sign of change.


In [ ]:
pakistan_heat_change <- climate_change_signal(
    pakistan_tasmax,
    season = list(1:3, 4:6, 7:9),
    uppert = HEAT_THRESHOLD_C,
    threshold = 0.80,
    bias.correction = FALSE
  )


## 6. Map ensemble change and model agreement

Positive values indicate more very hot pre-monsoon days in the future period. Stippling follows the package's model-agreement convention.


In [ ]:
heat_map <- plotting(
    pakistan_heat_change,
    ensemble = TRUE,
    plot_titles = "Change in days above 35 °C",
    temporal=TRUE
  )
  
heat_map



## 7. Optional: export the analysis raster

Keep export disabled during the live training unless participants need a GeoTIFF for GIS work.


In [ ]:
extract_raster(
    pakistan_heat_change,
    file.extension = "pakistan_pre_monsoon_heat_change.tif",
    ensemble = TRUE,
    path = "outputs"
  )


## 8. Optional extension: annual precipitation change

Repeat the workflow with `variable = "pr"`, set `aggr.m = "sum"` when loading, and call `climate_change_signal(..., percentage = TRUE, bias.correction = FALSE)`. Use `IPCC_palette(type = "pr", divergent = TRUE)` for the map.

Further examples: [CAVAanalytics introduction](https://un-fao.github.io/CAVAanalytics/articles/Introduction.html) and [advanced indicators](https://un-fao.github.io/CAVAanalytics/articles/more_advanced.html).


## Takeaway

CAVAanalytics is the stronger route when the goal is a country-scale, multi-model climate assessment with threshold indicators, model agreement, and publication-ready maps in a compact R workflow.
